# Structured — Feature Extraction

**Dataset:** Structured Retrospective (task-planning app)
**Stage:** 1 — Dataset-specific feature extraction (no cross-dataset harmonization)
**Input files:** `outputs/cleaned/structured_*.csv`
**Output files:** `outputs/features/structured/`

---

## Outputs produced

| File | Rows | Unit of observation | Contents |
|---|---|---|---|
| `structured_task_lifecycle_features.csv` | ~26,729 | one task | completion/deletion state, edit counts, censoring flag |
| `structured_participant_features.csv` | 61 | one participant (all behavioral) | volume, task-outcome composition, structure use, temporal patterns |
| `structured_survey_features.csv` | 61 | one participant (behavioral ∩ survey) | demographics, productivity/happiness ratings, ASRS-6 |

This is a prerequisite for every Structured model notebook (NeT, InT, StoP, DyS, SyD, StM, GaT) —
it is not a model notebook itself. See `outputs/reports/structured/structured_cas_modeling_plan.md`
for which model each output table feeds.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "features" / "structured"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUT)

## 1. Load cleaned files

In [ ]:
from src.features.structured_features import load_structured_files

beh, surv, linkage = load_structured_files(CLEANED)

print(f"Behavioral rows:        {len(beh):,}")
print(f"Behavioral participants: {beh['participant_id'].nunique():,}")
print(f"Survey rows:            {len(surv):,}")
print(f"Linkage rows:           {len(linkage):,}")

## 2. Build task-lifecycle features

Groups by `task_id` and reconstructs each task's observed lifecycle: completion state
(last-transition-wins between `task_completed`/`task_uncompleted` tags), deletion, edit counts
per change type, and whether the task's creation event is visible in this window at all.

In [ ]:
from src.features.structured_features import build_task_lifecycle_features

print("Building task-lifecycle features...")
task_feats = build_task_lifecycle_features(beh)
print(f"Shape: {task_feats.shape}")
task_feats.head(3)

## 3. Critical structural check — left-censored tasks

Most tasks in this window do NOT have a visible `Task Created` event — they already existed before
the 2026-03-01 extraction start (these are existing users with pre-existing task histories, per the
"retrospective" framing). This limits full-lifecycle (created→...→resolved) reconstruction to a
minority of tasks; transition-level analysis (state X→state Y within the window) remains valid for
all tasks regardless of censoring.

In [ ]:
print("has_creation_event distribution:")
print(task_feats['has_creation_event'].value_counts())
print()
print(f"Left-censored: {(~task_feats['has_creation_event']).mean()*100:.1f}%")
print()
print("first_event_type distribution (what state we first observe a task in):")
print(task_feats['first_event_type'].value_counts())

## 4. Task lifecycle status distribution

In [ ]:
print(task_feats['lifecycle_status'].value_counts())
print()
print("As share of all tasks:")
print(task_feats['lifecycle_status'].value_counts(normalize=True).round(4))
print()
print("Ever-uncompleted (completed then undone) among ever-completed tasks:")
ever_completed = task_feats[task_feats['ever_completed']]
print(f"{ever_completed['ever_uncompleted'].sum()} / {len(ever_completed)} "
      f"({ever_completed['ever_uncompleted'].mean()*100:.1f}%)")

## 5. Save Task-Lifecycle Features

In [ ]:
task_feats.to_csv(OUT / "structured_task_lifecycle_features.csv", index=False)
print(f"Saved: structured_task_lifecycle_features.csv  ({task_feats.shape})")

## 6. Build Participant Features

Aggregates task-lifecycle outcomes (completion/deletion/unresolved shares, task-type composition,
reschedule rate, structure use) with event-level temporal features (peak hour, weekend/evening share,
hour entropy — reused from the shared `behavioral_features` utilities used by DS1/DS4/DS5/DS6).

In [ ]:
from src.features.structured_features import build_structured_participant_features

print("Building participant features...")
participant_feats = build_structured_participant_features(beh, task_feats)
print(f"Shape: {participant_feats.shape}")
participant_feats.head(3)

## 7. Sanity Checks — Participant Features

In [ ]:
print("Task outcome composition (mean across participants):")
print(participant_feats[['pct_tasks_completed', 'pct_tasks_deleted', 'pct_tasks_unresolved']].mean().round(3))
print()
print("Sum of the three shares should be ~1.0 per participant:")
check = (participant_feats['pct_tasks_completed'] + participant_feats['pct_tasks_deleted']
         + participant_feats['pct_tasks_unresolved'])
print(check.describe())
print()
print("Task type composition (mean):")
print(participant_feats[['share_task_type_timed', 'share_task_type_all_day', 'share_task_type_inbox']].mean().round(3))
print()
print("Left-censoring varies substantially by participant (min/median/max):")
print(participant_feats['pct_tasks_left_censored'].describe())

## 8. Save Participant Features

In [ ]:
participant_feats.to_csv(OUT / "structured_participant_features.csv", index=False)
print(f"Saved: structured_participant_features.csv  ({participant_feats.shape})")

## 9. Build Survey Features (behavior-linked participants)

Restricted to the 61 participants with behavioral data. Rows are **not** pre-filtered on
`consent_given`/`is_finished` — callers must filter explicitly (same convention as
`structured_survey_clean.csv`).

In [ ]:
from src.features.structured_features import build_structured_survey_features

behavioral_ids = set(beh['participant_id'].unique())
survey_feats = build_structured_survey_features(surv, behavioral_ids)
print(f"Shape: {survey_feats.shape}")
print()
print("Consent/completion status among behavior-linked participants:")
print(survey_feats[['consent_given', 'is_finished']].value_counts(dropna=False))

## 10. Construct-Validity Check — Self-Reported vs. Behaviorally-Derived Completion

In [ ]:
usable = survey_feats[survey_feats['consent_given'] & survey_feats['is_finished']].copy()
merged = usable.merge(participant_feats[['participant_id', 'pct_tasks_completed']], on='participant_id', how='left')
merged['pct_planned_tasks_completed_frac'] = merged['pct_planned_tasks_completed_num'] / 100.0

valid = merged.dropna(subset=['pct_planned_tasks_completed_frac', 'pct_tasks_completed'])
print(f"N with both self-reported and behavioral completion measures: {len(valid)}")
if len(valid) >= 3:
    corr = valid['pct_planned_tasks_completed_frac'].corr(valid['pct_tasks_completed'])
    print(f"Correlation (self-reported %% planned tasks completed vs. behaviorally-derived completion rate): r={corr:.3f}")
else:
    print("Too few participants for a meaningful correlation.")
print()
print("This is the construct-validity check flagged in structured_cas_modeling_plan.md §4 — ")
print("does self-reported planning behavior match logged behavior? Exploratory only given small N.")

## 11. Save Survey Features

In [ ]:
survey_feats.to_csv(OUT / "structured_survey_features.csv", index=False)
print(f"Saved: structured_survey_features.csv  ({survey_feats.shape})")

## 12. Output Summary

In [ ]:
print("=" * 60)
print("Structured Feature Extraction -- Output Summary")
print("=" * 60)

for fname, df in [
    ("structured_task_lifecycle_features.csv", task_feats),
    ("structured_participant_features.csv", participant_feats),
    ("structured_survey_features.csv", survey_feats),
]:
    n_features = df.shape[1] - 1
    print(f"  {fname}")
    print(f"    {len(df):,} rows x {n_features} features")

print()
print("""
=== KEY FINDING FROM THIS STAGE ===
60.6%% of tasks are left-censored (no visible Task Created event) -- these existing
users already had task histories before the 2026-03-01 extraction start. This was
NOT caught during inspection/cleaning; it only surfaced when reconstructing
per-task lifecycles here. It materially affects Network Theory / System Dynamics
framing: creation-channel analysis is limited to ~39%% of tasks, but state-to-state
transition analysis remains valid for all tasks.
""")